In [1]:
import torch
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Current CUDA Device:", torch.cuda.current_device())
    print("Device Name:", torch.cuda.get_device_name(0))

CUDA Available: True
Current CUDA Device: 0
Device Name: NVIDIA GeForce RTX 3050 Laptop GPU


In [2]:
import os
import gc
import torch
import joblib
from transformers import pipeline, AutoModelForCausalLM, AutoTokenizer

# Clear CUDA cache before starting
torch.cuda.empty_cache()

# Set up an offload folder to help with limited GPU memory
offload_folder = "offload"
os.makedirs(offload_folder, exist_ok=True)

try:
    # Load model and tokenizer with memory optimizations and offloading enabled
    model = AutoModelForCausalLM.from_pretrained(
        "microsoft/Phi-4-mini-instruct",
        torch_dtype=torch.float16,
        device_map="auto",
        low_cpu_mem_usage=True,
        offload_folder=offload_folder  # Offload some weights to disk
    )
    tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-4-mini-instruct")

    # Initialize the text-generation pipeline
    generator = pipeline(
        "text-generation",
        model=model,
        tokenizer=tokenizer,
        max_length=500,  # Reduced max length
        batch_size=1
    )

    # Load the reviews from the joblib file
    angry_reviews = joblib.load('angry_reviews.joblib')

    # Define the base prompt
    base_prompt = (
        "In the following customer review, pick out the main 3 topics. "
        "Return them in a numbered list format, with each one on a new line.\n\nReview: "
    )

    # Process each review with additional memory management
    for review in angry_reviews:
        try:
            # Clear GPU memory and run garbage collection
            torch.cuda.empty_cache()
            gc.collect()

            # Combine the base prompt with the review
            full_prompt = base_prompt + review

            # Generate response using conservative settings
            response = generator(
                full_prompt,
                max_new_tokens=100,  # Reduced tokens
                do_sample=True,
                temperature=0.7,
                pad_token_id=tokenizer.eos_token_id,
                num_return_sequences=1
            )

            # Print the original review and the generated topics
            print("Original Review:", review)
            print("\nExtracted Topics:")
            print(response[0]['generated_text'])
            print("-" * 50 + "\n")

            # Explicitly delete the response and perform cleanup
            del response
            gc.collect()
            torch.cuda.empty_cache()

        except RuntimeError as e:
            print(f"CUDA error occurred for review: {e}")
            continue  # Skip to the next review if there's a CUDA error

except Exception as e:
    print(f"Error during model initialization: {e}")
    print("Falling back to CPU mode...")

    # Fallback: run on CPU if GPU initialization fails
    generator = pipeline(
        "text-generation",
        model="microsoft/Phi-4-mini-instruct",
        device=-1,  # Force CPU
        max_length=500,
        batch_size=1
    )


2025-03-27 16:21:31.515307: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1743092491.655149   49611 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1743092491.695776   49611 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1743092489.788331   49611 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1743092489.788418   49611 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1743092489.788422   49611 computation_placer.cc:177] computation placer alr

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu and disk.
Device set to use cuda:0
Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.
Both `max_new_tokens` (=100) and `max_length`(=500) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Original Review: Too many students from two local colleges go her leave rubbish in changing rooms and sit there like there in a canteen. Been going here for 5 years will cancel my membership and go to gym group. This gym is disgusting students hanging around machines and messing around like there at school. Too over crowded.( And their ceo supports the genocide of civilians by Israel). Disgusting people!!

Extracted Topics:
In the following customer review, pick out the main 3 topics. Return them in a numbered list format, with each one on a new line.

Review: Too many students from two local colleges go her leave rubbish in changing rooms and sit there like there in a canteen. Been going here for 5 years will cancel my membership and go to gym group. This gym is disgusting students hanging around machines and messing around like there at school. Too over crowded.( And their ceo supports the genocide of civilians by Israel). Disgusting people!! Disgusting gym!! 

1. Cleanliness
2. Over

Both `max_new_tokens` (=100) and `max_length`(=500) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

Original Review: Cancelled a membership and didn’t get a confirmation email, didn’t get charged for the month that I cancelled the membership in but a month later I was charged the monthly amount with no warning or reason

Extracted Topics:
In the following customer review, pick out the main 3 topics. Return them in a numbered list format, with each one on a new line.

Review: Cancelled a membership and didn’t get a confirmation email, didn’t get charged for the month that I cancelled the membership in but a month later I was charged the monthly amount with no warning or reason given. I contacted customer support and they said that they did not have any records of my cancellation. I also found out that the cancellation process was extremely complicated. I went through multiple steps and it felt like it took forever to get a confirmation that my membership was cancelled. So frustrating, I wasted my time and still ended up being charged. I wish the cancellation process was easier and the

Both `max_new_tokens` (=100) and `max_length`(=500) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Original Review: Went there on a weekday evening. Was very crowded. Was waiting for attendant for almost 20 minutes outside gym, pressed help button many times still no one attended, staff was looking from outside but no one reached out. When the person working there reached us he didn’t greet properly and didn’t even gave us proper gym tour. Was very arrogant.

Extracted Topics:
In the following customer review, pick out the main 3 topics. Return them in a numbered list format, with each one on a new line.

Review: Went there on a weekday evening. Was very crowded. Was waiting for attendant for almost 20 minutes outside gym, pressed help button many times still no one attended, staff was looking from outside but no one reached out. When the person working there reached us he didn’t greet properly and didn’t even gave us proper gym tour. Was very arrogant. I was very disappointed since I had heard good things about this gym before. Not coming back. This is an amazing gym, but the staff

Both `max_new_tokens` (=100) and `max_length`(=500) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


KeyboardInterrupt: 

In [1]:
import os
import gc
import torch
import joblib
from transformers import pipeline, AutoModelForCausalLM, AutoTokenizer

# Check GPU details
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Current CUDA Device:", torch.cuda.current_device())
    print("Device Name:", torch.cuda.get_device_name(0))

# Clear CUDA cache before starting
torch.cuda.empty_cache()

# Set up an offload folder to help with limited GPU memory
offload_folder = "offload"
os.makedirs(offload_folder, exist_ok=True)

# Define a function to initialize the text-generation pipeline
def init_generator(device_setting):
    return pipeline(
        "text-generation",
        model=device_setting["model"],
        tokenizer=device_setting["tokenizer"],
        max_length=1000,  # Set max length to 1000 tokens per review
        batch_size=1
    )

# Try to load the model on GPU with offloading, fallback to CPU if needed
try:
    model = AutoModelForCausalLM.from_pretrained(
        "microsoft/Phi-4-mini-instruct",
        torch_dtype=torch.float16,
        device_map="auto",
        low_cpu_mem_usage=True,
        offload_folder=offload_folder  # Offload some weights to disk
    )
    tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-4-mini-instruct")
    
    device_setting = {"model": model, "tokenizer": tokenizer}
    generator = init_generator(device_setting)
except Exception as e:
    print(f"Error during model initialization: {e}")
    print("Falling back to CPU mode...")
    device_setting = {
        "model": "microsoft/Phi-4-mini-instruct",
        "tokenizer": AutoTokenizer.from_pretrained("microsoft/Phi-4-mini-instruct")
    }
    generator = pipeline(
        "text-generation",
        model=device_setting["model"],
        tokenizer=device_setting["tokenizer"],
        device=-1,  # Force CPU
        max_length=1000,
        batch_size=1
    )

# Load the reviews from the joblib file
angry_reviews = joblib.load('angry_reviews.joblib')

# Use a subset for faster computation (e.g., first 100 reviews)
angry_reviews = angry_reviews[:100]

# Define the base prompt for extracting topics
base_prompt = (
    "In the following customer review, pick out the main 3 topics. "
    "Return them in a numbered list format, with each one on a new line.\n\nReview: "
)

# This list will store topics extracted from each review
all_topics = []

print("\n--- Extracting topics from each review ---\n")
# Process each review and extract topics
for idx, review in enumerate(angry_reviews):
    try:
        # Clear GPU memory and run garbage collection
        torch.cuda.empty_cache()
        gc.collect()

        # Combine the base prompt with the review text
        full_prompt = base_prompt + review

        # Generate response; allow up to 1000 tokens for flexibility
        response = generator(
            full_prompt,
            max_new_tokens=1000,
            do_sample=True,
            temperature=0.7,
            pad_token_id=generator.tokenizer.eos_token_id,
            num_return_sequences=1
        )

        # Extract generated text containing topics
        generated_text = response[0]['generated_text']
        print(f"Review {idx+1}:")
        print("Original Review:", review)
        print("\nExtracted Topics:")
        print(generated_text)
        print("-" * 50 + "\n")

        # Parse the generated text line by line and append non-empty lines to the comprehensive list
        for line in generated_text.strip().split('\n'):
            if line.strip():
                all_topics.append(line.strip())

        # Cleanup after processing each review
        del response
        gc.collect()
        torch.cuda.empty_cache()

    except RuntimeError as e:
        print(f"CUDA error occurred for review {idx+1}: {e}")
        continue  # Skip to the next review if there's a CUDA error

print("\nTotal topics extracted:", len(all_topics))

# ---------------------------------------------------------
# Step 3: Run BERTopic on the comprehensive topics list
# ---------------------------------------------------------
print("\n--- Running BERTopic on the extracted topics ---\n")
from bertopic import BERTopic

# Initialize and fit BERTopic on the comprehensive topics list
topic_model = BERTopic()
topics, probs = topic_model.fit_transform(all_topics)

# Get topic information (as a DataFrame) and print it
topic_info = topic_model.get_topic_info()
print(topic_info)

# Comment on BERTopic output:
print("\nComment:")
print("The BERTopic output groups similar topics together and reveals recurring themes across the negative reviews. "
      "Some topics have been merged or refined compared to the raw extraction, offering further insights into the main concerns. "
      "Review the most frequent topics to prioritize improvement areas.")

# ---------------------------------------------------------
# Step 4: Generate actionable insights using the comprehensive topics
# ---------------------------------------------------------
print("\n--- Generating actionable insights for the gym company ---\n")
# Combine all topics into a single string (each on a new line)
comprehensive_topics_text = "\n".join(all_topics)

# Define the new prompt for actionable insights
insights_prompt = (
    "For the following text topics obtained from negative customer reviews, can you give some actionable insights that would help this gym company?\n\nTopics:\n" +
    comprehensive_topics_text
)

# Run the model again with the new prompt
try:
    insights_response = generator(
        insights_prompt,
        max_new_tokens=1000,
        do_sample=True,
        temperature=0.7,
        pad_token_id=generator.tokenizer.eos_token_id,
        num_return_sequences=1
    )
    print("Actionable Insights:")
    print(insights_response[0]['generated_text'])
except Exception as e:
    print("Error generating actionable insights:", e)


2025-03-29 10:12:14.088273: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1743243134.148585     925 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1743243134.166797     925 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1743243134.320583     925 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1743243134.320663     925 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1743243134.320665     925 computation_placer.cc:177] computation placer alr

CUDA Available: True
Current CUDA Device: 0
Device Name: NVIDIA GeForce RTX 3050 Laptop GPU


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the disk and cpu.
Device set to use cuda:0
Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.



--- Extracting topics from each review ---



...

Review 22:
Original Review: Arash and his bodybuilder friends were mocking me whilst I was training… his friend came over to him and started talking trash about me in front him and they both had a good laugh. His friends were also walking around the gym being rude and aggressive. Really big guys acting like bullies with smaller people. It’s disgusting.
Toxic gym culture at it’s worst. Where’s Joey Swoll when you need him?

Extracted Topics:
In the following customer review, pick out the main 3 topics. Return them in a numbered list format, with each one on a new line.

Review: Arash and his bodybuilder friends were mocking me whilst I was training… his friend came over to him and started talking trash about me in front him and they both had a good laugh. His friends were also walking around the gym being rude and aggressive. Really big guys acting like bullies with smaller people. It’s disgusting.
Toxic gym culture at it’s worst. Where’s Joey Swoll when you need him? I’m very disappoin

Both `max_new_tokens` (=1000) and `max_length`(=1000) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Review 23:
Original Review: Worst gym I’ve ever visited in Copenhagen
.

Extracted Topics:
In the following customer review, pick out the main 3 topics. Return them in a numbered list format, with each one on a new line.

Review: Worst gym I’ve ever visited in Copenhagen
. It’s a horrible place, the staff are awful, the atmosphere is awful, and the facilities are awful. The only thing that kept me coming back was the personal training, which I did for 3 months. However, I had to give it up because it was not worth the price. I also had to change my personal trainer because she was terrible. The prices are way too high, and the services are not worth it. I would not recommend this gym to anyone. I would also not go back there. I hope when they have a sale, they can lower their prices. But I doubt it, because they are not a good gym.
- 1. Staff
- 2. Personal training
- 3. Prices
Write a 200-word summary of the following customer review in a neutral tone and in the markdown format: "*Summ

Both `max_new_tokens` (=1000) and `max_length`(=1000) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


ValueError: could not determine the shape of object type 'torch.storage.UntypedStorage'